# Backend para la Carga de Datos (FastAPI) - PRUEBA/EJEMPLO
Este notebook levanta un servidor FastAPI en el puerto 8000 para recibir el archivo `.csv` desde el frontend de React/Next.js.

In [ ]:
%pip install fastapi uvicorn python-multipart pandas nest-asyncio

In [7]:
import nest_asyncio
import uvicorn
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.middleware.cors import CORSMiddleware
import pandas as pd
import io

# Permitir asincronía en Jupyter
nest_asyncio.apply()

app = FastAPI(title="CreditGuard API")

# Configuración de CORS para permitir solicitudes desde el frontend
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # Permite todos los orígenes en desarrollo
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Variable global para mantener el dataset en memoria
global_df = None

@app.post("/carga")
async def cargar_csv(archivo: UploadFile = File(...)):
    global global_df
    
    if not archivo.filename.endswith('.csv'):
        raise HTTPException(status_code=400, detail="El archivo debe ser un .csv")
        
    try:
        # Leer el contenido del archivo
        contents = await archivo.read()
        
        # Cargar el CSV en un DataFrame de pandas
        df = pd.read_csv(io.StringIO(contents.decode('utf-8')))
        
        # Guardar en memoria para usarlo en la siguiente fase (Limpieza)
        global_df = df
        
        # Guardar una copia localmente por seguridad
        df.to_csv("dataset_original.csv", index=False)
        
        # Retornar el resultado estructurado según lo espera el frontend
        return {
            "filas": int(df.shape[0]),
            "columnas": int(df.shape[1]),
            "nombre_archivo": archivo.filename,
            "mensaje": "Archivo cargado exitosamente"
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Error al procesar el archivo: {str(e)}")


### Ejecutar el servidor
Ejecuta la siguiente celda para iniciar el backend. Se quedará en ejecución escuchando en `http://localhost:8000`.
Para detenerlo, detén la ejecución de la celda (el botón cuadrado en la barra superior de la celda).

In [ ]:
if __name__ == "__main__":
    print("Iniciando servidor FastAPI en http://localhost:8000 ...")
    
    # En Jupyter (especialmente con Python 3.13), uvicorn.run lanza un error de 
    # "asyncio.run() cannot be called from a running event loop". 
    # La forma correcta de correrlo aquí es usando await en la configuración de uvicorn:
    config = uvicorn.Config(app, host="0.0.0.0", port=8000)
    server = uvicorn.Server(config)
    await server.serve()

Iniciando servidor FastAPI en http://localhost:8000 ...


INFO:     Started server process [23092]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:57403 - "POST /carga HTTP/1.1" 200 OK
INFO:     127.0.0.1:52443 - "POST /limpieza HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:59555 - "POST /carga HTTP/1.1" 200 OK
